# 02 Interpret Results

Load a completed sweep and regenerate figures + robust summaries.

This notebook now detects chain-bandit runs and writes the new mean/variance sensitivity CSV automatically.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Project root:", PROJECT_ROOT)


In [ ]:
import pandas as pd

from ess_ope.evaluation.summary import (
    PaperClaimConfig,
    SummaryConfig,
    build_chain_bandit_sensitivity_summary,
    build_condition_summary,
    build_estimator_summary,
    build_paper_claim_summary,
    build_paper_claims_table,
)
from ess_ope.plotting.benchmark_figures import generate_benchmark_figures

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 220)

latest_dir = (PROJECT_ROOT / "results/latest").resolve()
pq = latest_dir / "sweep_results.parquet"
csv = latest_dir / "sweep_results.csv"

if pq.exists():
    df = pd.read_parquet(pq)
    source = pq
elif csv.exists():
    df = pd.read_csv(csv)
    source = csv
else:
    raise FileNotFoundError("No sweep_results in results/latest")

print("Loaded:", source)
print("Rows:", len(df))
if "env_name" in df.columns:
    print("env_name:", sorted(df.env_name.unique().tolist()))
print("alphas:", sorted(df.alpha.unique().tolist()))
print("betas:", sorted(df.beta.unique().tolist()))
print("K:", sorted(df.K.unique().tolist()))
if "transition_strength" in df.columns:
    print("transition_strengths:", sorted(df.transition_strength.unique().tolist()))
if "reward_mean_scale" in df.columns:
    print("reward_mean_scales:", sorted(df.reward_mean_scale.unique().tolist()))
if "reward_std" in df.columns:
    print("reward_stds:", sorted(df.reward_std.unique().tolist()))
if "chain_variant" in df.columns:
    print("chain_variants:", sorted(df.chain_variant.unique().tolist()))


In [ ]:
fig_dir = latest_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

benchmark_report = generate_benchmark_figures(df, output_dir=fig_dir, fixed_alpha=None, fixed_beta=0.0)
summary_cfg = SummaryConfig(bootstrap_samples=16000, bootstrap_max_points=200000, show_progress=True)
estimator_summary = build_estimator_summary(df, config=summary_cfg)
condition_summary = build_condition_summary(df, show_progress=True)
chain_bandit_sensitivity = build_chain_bandit_sensitivity_summary(df)
paper_claims = build_paper_claims_table(df, estimator_summary, benchmark_report, config=PaperClaimConfig())
paper_claim_summary = build_paper_claim_summary(paper_claims)

benchmark_report.to_csv(fig_dir / "benchmark_report.csv", index=False)
estimator_summary.to_csv(fig_dir / "estimator_summary.csv", index=False)
condition_summary.to_csv(fig_dir / "condition_summary.csv", index=False)
if not chain_bandit_sensitivity.empty:
    chain_bandit_sensitivity.to_csv(fig_dir / "chain_bandit_sensitivity_summary.csv", index=False)
paper_claims.to_csv(fig_dir / "paper_claims.csv", index=False)
paper_claim_summary.to_csv(fig_dir / "paper_claim_summary.csv", index=False)

print("Saved to:", fig_dir)
paper_claims


In [ ]:
if not chain_bandit_sensitivity.empty:
    chain_bandit_sensitivity
else:
    paper_claim_summary


In [ ]:
from IPython.display import Image, display

for name in [
    "benchmark_fig1_ess_vs_error_by_estimator.png",
    "benchmark_fig2_same_ess_different_error.png",
    "benchmark_fig3_ess_changes_error_stability.png",
    "benchmark_fig4_fan_estimate_vs_ess.png",
    "benchmark_fig5_mean_variance_sensitivity.png",
]:
    p = fig_dir / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))
